# Random Forest — multibranch-style sequence classifier

This notebook uses the cleaned `base_utils_qwen.py` and trains a sequence-level Random Forest.

Local quick runs use `data/sample.csv` (37 sequences). Set `use_sample_data = False` for full `train.csv`.

Switch `search_mode` between `'grid'` and `'bayesian'` in the config cell.

Style:
- configuration
- data loading
- split
- estimator
- parameter search (grid or Bayesian)
- holdout evaluation
- save results


In [1]:
import os
import sys
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Local path routing (works from notebooks/ or project root)
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# Kaggle path routing
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception:
    pass

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.metrics import f1_score, make_scorer

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

try:
    from src import data_utils
    from src.base_utils_qwen import (
        SequenceExtractor,
        RandomForestSequenceClassifier,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
    )
    print('Imports loaded from src/')
except ImportError:
    import data_utils
    from base_utils_qwen import (
        SequenceExtractor,
        RandomForestSequenceClassifier,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
    )
    print('Imports loaded from flat src path')


Imports loaded from flat src path


In [2]:
# Install optional search / feature dependencies if missing (safe to re-run)
for package_name, import_name in [
    ('scikit-optimize', 'skopt'),
    ('PyWavelets', 'pywt'),
]:
    try:
        __import__(import_name)
    except ImportError:
        import subprocess
        import sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])
        print(f'Installed {package_name}')

try:
    import skopt
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
    print(f'scikit-optimize {skopt.__version__} ready')
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False
    print('scikit-optimize unavailable')

try:
    import pywt
    print(f'PyWavelets {pywt.__version__} ready')
except ImportError:
    print('PyWavelets unavailable')

scikit-optimize 0.10.2 ready
PyWavelets 1.8.0 ready


In [3]:
TARGET_COL = 'bfrb'

# Use sample.csv for a quick smoke test; set False for full train.csv
use_sample_data = True
sample_file = 'sample.csv'  # also available: eg.csv

search_mode = 'grid'  # 'grid' or 'bayesian' for quick smoke test use 'grid'
random_state = 42
n_splits = 1          # 1 -> GroupShuffleSplit; >=2 -> GroupKFold
cv_test_size = 0.3
train_size = 0.7
n_iter = 5            # Bayesian iterations for a quick smoke test
verbose = 1
error_score = 'raise'  # 'raise' or 'warn'

results_dir = Path('results_rf_multibranch_style')
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M')

if TARGET_COL == 'bfrb':
    scorer = competition_scorer
else:
    scorer = make_scorer(f1_score, average='macro', zero_division=0)

search_mode = str(search_mode).lower()
if search_mode in ('bayes', 'bayesian'):
    search_mode = 'bayesian'
elif search_mode != 'grid':
    raise ValueError("search_mode must be 'grid' or 'bayesian'")

if n_splits <= 1:
    cv_object = GroupShuffleSplit(
        n_splits=1,
        test_size=cv_test_size,
        random_state=random_state,
    )
else:
    cv_object = GroupKFold(n_splits=n_splits)


In [4]:
data_root = data_utils.find_data_root()
sample_path = data_root / sample_file

if use_sample_data and sample_path.exists():
    raw_train_df = pd.read_csv(sample_path)
    print(f'Using {sample_file}: {raw_train_df["sequence_id"].nunique()} sequences')
else:
    raw_train_df = pd.read_csv(data_root / 'train.csv')
    print(f'Using train.csv: {raw_train_df["sequence_id"].nunique()} sequences')

train_demo_df = pd.read_csv(data_root / 'train_demographics.csv')

train_df = raw_train_df.set_index('row_id').copy(deep=True)

train_df['gesture'] = train_df['gesture'].fillna('non_bfrb').astype(str)
train_df['orientation'] = train_df['orientation'].fillna('Unknown').astype(str)
train_df['is_target'] = train_df['sequence_type'].eq('Target').astype(int)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'].astype(bool), 'non_bfrb')

train_df['gesture_position'] = train_df['gesture'].str.split(' - ').str[0]
train_df['gesture_action'] = train_df['gesture'].str.split(' - ').str[-1]

if TARGET_COL not in train_df.columns:
    train_df[TARGET_COL] = train_df['gesture_action']

train_df[TARGET_COL] = train_df[TARGET_COL].fillna('non_bfrb').astype(str)


Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data
Using train.csv: 8151 sequences


In [5]:
try:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df,
        train_pct=train_size,
        test_pct=min(0.2, 1 - train_size),
        random_state=random_state,
    )
except Exception:
    seq_df = train_df[['sequence_id', 'is_target', TARGET_COL]].drop_duplicates('sequence_id').sort_values('sequence_id')
    gss = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(seq_df, groups=seq_df['sequence_id']))

    train_seqs = seq_df.iloc[train_idx]['sequence_id']
    test_seqs = seq_df.iloc[test_idx]['sequence_id']

    train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)].copy()
    hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)].copy()

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()

y_train = X_train[['sequence_id', 'is_target', TARGET_COL]].copy()
y_test = X_test[['sequence_id', 'is_target', TARGET_COL]].copy()

groups = X_train['sequence_id'].astype(str)

print('Train sequences:', X_train['sequence_id'].nunique())
print('Test sequences:', X_test['sequence_id'].nunique())


Train: 3879 seqs | 47.6%
Test:  966 seqs  | 11.9%
Train sequences: 3879
Test sequences: 966


In [6]:
extractor = SequenceExtractor(
    acc_modes='raw|velocity|jerk',
    rotation_modes='quaternion|angular_velocity',
    tof_modes='pooled_stats|sensor_stats',
    thm_modes='centered_diff',
    motion_filter_mode=None,
    use_dead_reckoning=False,
    compute_dt=True,
    interp_mode='linear',
    padding_value=0.0,
    chunk_window_size=None,
    chunk_stride=None,
    output_format='frame',
    frame_stats='mean,std,min,max,last',
    add_global_context=False,
    resample_modalities=False,
)

estimator = RandomForestClassifier(
    n_estimators=300,
    random_state=random_state,
    n_jobs=-1,
    class_weight='balanced_subsample',
)

model = RandomForestSequenceClassifier(
    primary_target=TARGET_COL,
    extractor=extractor,
    estimator=estimator,
    random_state=random_state,
)


In [7]:
# ============================================================
# GRID SEARCH SPACE
# Practical compact grid. Bayesian space does the full exploration.
# ============================================================

GRID_PARAM_SPACE = {
    # ------------------------------------------------------------
    # EXTRACTOR: main sensor-feature domains
    # ------------------------------------------------------------
    'extractor__acc_modes': [
        'raw',
        # 'smoothed|velocity|displacement|jerk',
    ],

    'extractor__rotation_modes': [
        'raw',
        # 'quaternion|euler|angular_velocity',
    ],

    'extractor__tof_modes': [
        # 'pooled_stats|sensor_stats',
        'raw',
    ],

    'extractor__thm_modes': [
        'centered_diff',
        # 'centered',
    ],

    'extractor__frame_stats': [
        'mean,std,min,max,last',
        # 'mean,std,min,max,last,first,rms',
    ],

    # ------------------------------------------------------------
    # EXTRACTOR: fixed preprocessing for stable RF smoke/grid runs
    # ------------------------------------------------------------
    'extractor__motion_filter_mode': [None],
    'extractor__use_dead_reckoning': [False],
    'extractor__dead_reckoning_detrend': [False],
    'extractor__kalman_process_noise': [1e-3],
    'extractor__kalman_measurement_noise': [1e-1],

    'extractor__window_size': [20],
    'extractor__smooth_alpha': [None],
    'extractor__clip_value': [None],
    'extractor__interp_mode': ['linear'],

    # ------------------------------------------------------------
    # EXTRACTOR: fixed frame-output safety params
    # ------------------------------------------------------------
    'extractor__output_format': ['frame'],
    'extractor__padding_value': [0.0],
    'extractor__maxlen': [160],
    'extractor__chunk_window_size': [None],
    'extractor__chunk_stride': [None],
    'extractor__add_global_context': [False],
    'extractor__resample_modalities': [False],
    'extractor__compute_dt': [True],

    'extractor__imu_native_sampling_rate': [20],
    'extractor__rot_native_sampling_rate': [20],
    'extractor__tof_native_sampling_rate': [5],
    'extractor__thm_native_sampling_rate': [5],

    'extractor__imu_target_sampling_rate': [20],
    'extractor__rot_target_sampling_rate': [20],
    'extractor__tof_target_sampling_rate': [5],
    'extractor__thm_target_sampling_rate': [5],

    # ------------------------------------------------------------
    # EXTRACTOR: STFT & CWT (Time-Frequency Domains) - FIXED FOR GRID
    # ------------------------------------------------------------
    'extractor__stft_nperseg': [32],
    'extractor__stft_noverlap': [None],
    'extractor__stft_window_type': ['hann'],
    'extractor__stft_use_log_scale': [True],

    'extractor__cwt_wavelet': ['morl'],
    'extractor__cwt_max_scale': [128],
    'extractor__cwt_n_scales': [32],
    'extractor__cwt_use_log_scale': [True],

    # ------------------------------------------------------------
    # RANDOM FOREST ESTIMATOR
    # ------------------------------------------------------------
    'estimator__n_estimators': [10],
    'estimator__criterion': ['gini'],
    'estimator__max_depth': [None],
    'estimator__min_samples_split': [2],
    'estimator__min_samples_leaf': [1],
    'estimator__max_features': ['sqrt'],
    'estimator__bootstrap': [True],
    'estimator__class_weight': [None],
}


# ============================================================
# BAYESIAN SEARCH SPACE
# Full exploration space.
# ============================================================

if SKOPT_AVAILABLE:
    try:
        BAYESIAN_PARAM_SPACE = {
            # --------------------------------------------------------
            # EXTRACTOR: sensor-feature domains
            # --------------------------------------------------------
            'extractor__acc_modes': Categorical([
                'raw',
                'raw|velocity',
                'smoothed|velocity|displacement|jerk',
            ]),

            'extractor__rotation_modes': Categorical([
                'quaternion|euler|angular_velocity',
                'quaternion|angular_velocity|delta_euler|rot6d',
            ]),

            'extractor__tof_modes': Categorical([
                'pooled_stats|sensor_stats',
            ]),

            'extractor__thm_modes': Categorical([
                'centered_diff',
            ]),

            'extractor__frame_stats': Categorical([
                'mean,std,min,max,last',
                'mean,std,min,max,last,first,rms,abs_mean',
            ]),

            # --------------------------------------------------------
            # EXTRACTOR: preprocessing / filtering
            # --------------------------------------------------------
            'extractor__motion_filter_mode': Categorical([
                'extended_kalman',
            ]),

            'extractor__use_dead_reckoning': Categorical([True]),
            'extractor__dead_reckoning_detrend': Categorical([True]),

            'extractor__kalman_process_noise': Real(
                1e-5,
                1e-1,
                prior='log-uniform',
            ),

            'extractor__kalman_measurement_noise': Real(
                1e-3,
                1e1,
                prior='log-uniform',
            ),

            'extractor__window_size': Integer(5, 50),

            'extractor__smooth_alpha': Categorical([
                None,
                0.05,
                0.10,
                0.20,
                0.90,
            ]),

            'extractor__clip_value': Categorical([
                150.0,
            ]),

            'extractor__interp_mode': Categorical([
                'linear',
            ]),

            # --------------------------------------------------------
            # EXTRACTOR: STFT & CWT (Time-Frequency Domains) - BAYESIAN
            # --------------------------------------------------------
            'extractor__stft_nperseg': Categorical([16, 32, 64, 128]),
            'extractor__stft_noverlap': Categorical([None, 8, 16, 32, 64]),
            'extractor__stft_window_type': Categorical(['hann', 'hamming', 'blackman']),
            'extractor__stft_use_log_scale': Categorical([True, False]),

            'extractor__cwt_wavelet': Categorical(['morl', 'mexh', 'gaus1', 'gaus2']),
            'extractor__cwt_max_scale': Categorical([64, 128, 256]),
            'extractor__cwt_n_scales': Categorical([16, 32, 64]),
            'extractor__cwt_use_log_scale': Categorical([True, False]),

            # --------------------------------------------------------
            # EXTRACTOR: fixed frame-output safety params
            # --------------------------------------------------------
            'extractor__output_format': Categorical(['frame']),
            'extractor__padding_value': Categorical([0.0]),
            'extractor__maxlen': Categorical([150]),
            'extractor__chunk_window_size': Categorical([None, 100]),
            'extractor__chunk_stride': Categorical([None, 50]),
            'extractor__add_global_context': Categorical([True]),
            'extractor__compute_dt': Categorical([True]),

            'extractor__imu_native_sampling_rate': Categorical([100]),
            'extractor__rot_native_sampling_rate': Categorical([100]),
            'extractor__tof_native_sampling_rate': Categorical([20]),
            'extractor__thm_native_sampling_rate': Categorical([20]),

            # Target sampling rates (used when resample_modalities=True).
            # Bad combos fail fast via InvalidExtractorParams in base_utils_qwen.
            'extractor__imu_target_sampling_rate': Categorical([100]),
            'extractor__rot_target_sampling_rate': Categorical([100]),
            'extractor__tof_target_sampling_rate': Categorical([20]),
            'extractor__thm_target_sampling_rate': Categorical([20]),
            'extractor__resample_modalities': Categorical([ True]),

            # --------------------------------------------------------
            # RANDOM FOREST ESTIMATOR
            # --------------------------------------------------------
            'estimator__n_estimators': Integer(5, 1200),

            'estimator__criterion': Categorical([
                'gini',
                'entropy',
            ]),

            'estimator__max_depth': Categorical([
                None,
                10,
                20,
                30,
                50,
                80,
            ]),

            'estimator__min_samples_split': Integer(2, 20),
            'estimator__min_samples_leaf': Integer(1, 8),

            'estimator__max_features': Categorical([
                'sqrt',
                'log2',
                0.3,
                0.5,
                0.7,
                0.9,
            ]),

            'estimator__bootstrap': Categorical([True]),

            'estimator__class_weight': Categorical([
                'balanced'
            ]),

            'estimator__min_impurity_decrease': Real(0.0, 0.005),
        }

    except Exception:
        BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE
else:
    BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE

param_space = BAYESIAN_PARAM_SPACE if search_mode == 'bayesian' else GRID_PARAM_SPACE


In [8]:
if search_mode == 'bayesian':
    if not SKOPT_AVAILABLE:
        raise ImportError(
            "Bayesian search requires scikit-optimize. "
            "Install with: pip install scikit-optimize"
        )

    search = BayesSearchCV(
        estimator=model,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        random_state=random_state,
        verbose=verbose,
        return_train_score=True,
        error_score=error_score,
    )
else:
    search = GridSearchCV(
        estimator=model,
        param_grid=param_space,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        verbose=verbose,
        return_train_score=True,
        error_score=error_score,
    )

search.fit(X_train, y_train, groups=groups)

print('Best CV score:', search.best_score_)
print('Best params:', search.best_params_)


Fitting 1 folds for each of 1 candidates, totalling 1 fits
Best CV score: 0.5599058231820099
Best params: {'estimator__bootstrap': True, 'estimator__class_weight': None, 'estimator__criterion': 'gini', 'estimator__max_depth': None, 'estimator__max_features': 'sqrt', 'estimator__min_samples_leaf': 1, 'estimator__min_samples_split': 2, 'estimator__n_estimators': 10, 'extractor__acc_modes': 'raw', 'extractor__add_global_context': False, 'extractor__chunk_stride': None, 'extractor__chunk_window_size': None, 'extractor__clip_value': None, 'extractor__compute_dt': True, 'extractor__cwt_max_scale': 128, 'extractor__cwt_n_scales': 32, 'extractor__cwt_use_log_scale': True, 'extractor__cwt_wavelet': 'morl', 'extractor__dead_reckoning_detrend': False, 'extractor__frame_stats': 'mean,std,min,max,last', 'extractor__imu_native_sampling_rate': 20, 'extractor__imu_target_sampling_rate': 20, 'extractor__interp_mode': 'linear', 'extractor__kalman_measurement_noise': 0.1, 'extractor__kalman_process_noise

In [9]:
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)

eval_results = evaluate_holdout(
    y_test,
    y_pred,
    target_col=TARGET_COL,
    verbose=True,
)

print('Holdout competition score:', eval_results['competition_score'])

cv_df = pd.DataFrame(search.cv_results_)
cv_df.to_csv(results_dir / f'rf_multibranch_style_cv_{timestamp}.csv', index=False)

eval_results['results_df'].to_csv(
    results_dir / f'rf_multibranch_style_holdout_{timestamp}.csv',
    index=False,
)

pd.DataFrame(
    [
        {
            'best_score': search.best_score_,
            'best_params': str(search.best_params_),
            'holdout_score': eval_results['competition_score'],
        }
    ]
).to_csv(
    results_dir / f'rf_multibranch_style_best_{timestamp}.csv',
    index=False,
)



FINAL EVALUATION
Binary F1 (non_bfrb vs bfrb): 0.8774
BFRB Gesture Macro F1: 0.3591
COMPETITION SCORE: 0.6183

----------------------------------------
BFRB Gesture Classification Report
----------------------------------------
                          precision    recall  f1-score   support

   Above ear - pull hair       0.45      0.46      0.45        81
      Cheek - pinch skin       0.41      0.36      0.38        81
     Eyebrow - pull hair       0.45      0.32      0.37        81
     Eyelash - pull hair       0.41      0.32      0.36        81
Forehead - pull hairline       0.55      0.41      0.47        81
      Forehead - scratch       0.56      0.43      0.49        81
       Neck - pinch skin       0.36      0.30      0.32        81
          Neck - scratch       0.47      0.32      0.38        81
                non_bfrb       0.00      0.00      0.00         0

                accuracy                           0.36       648
               macro avg       0.41      0.

In [10]:
importances = pd.Series(
    best_model.estimator_.feature_importances_,
    index=best_model.extractor_.frame_feature_names_,
).sort_values(ascending=False)

print(importances.head(50))


tof_2_v47_raw_mean                  0.012980
tof_2_v31_raw_mean                  0.012366
tof_2_v62_raw_mean                  0.011102
tof_2_v30_raw_mean                  0.010849
tof_2_v55_raw_mean                  0.009470
tof_2_v53_raw_mean                  0.008698
tof_4_v55_raw_mean                  0.008375
tof_3_v19_raw_mean                  0.008319
tof_2_v28_raw_mean                  0.007346
tof_3_v15_raw_std                   0.007069
tof_2_v31_raw_last                  0.006838
tof_3_v23_raw_std                   0.006542
tof_3_v47_raw_mean                  0.005527
tof_3_v3_raw_mean                   0.004979
tof_3_v0_raw_min                    0.004847
tof_2_v23_raw_mean                  0.004696
tof_2_v46_raw_mean                  0.004625
acc_y_stft_band_low_mean_min        0.004248
tof_2_v47_raw_std                   0.003890
tof_2_v63_raw_mean                  0.003889
tof_3_v5_raw_std                    0.003819
tof_3_v55_raw_mean                  0.003796
tof_3_v28_